# v7.7 Phase 1: PyCaret compare_models 模型筛选

目标：用 PyCaret 一次性对比多个回归模型，选出 R2 最高的 top-5

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

# 切换到 repo 根目录
os.chdir(os.path.expanduser('~/Public/QuantNodes'))
print(f'工作目录: {os.getcwd()}')

from pycaret.regression import *

工作目录: /home/ll/Public/QuantNodes


In [2]:
# 加载训练面板
panel = pd.read_parquet('data/high_freq_macro/v7_7_train_panel.parquet')
feat_cols = [c for c in panel.columns if c.startswith('f')]
print(f'原始数据: {panel.shape}')
print(f'因子列: {len(feat_cols)}')

原始数据: (15984, 42)
因子列: 39


In [3]:
# 清理 inf / 极大值
df = panel[feat_cols + ['target_raw']].dropna(subset=['target_raw']).copy()

for col in feat_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    p99 = df[col].abs().quantile(0.999)
    if p99 > 0:
        df[col] = df[col].clip(-p99, p99)
df[feat_cols] = df[feat_cols].fillna(0)

# 重命名 target
df = df.rename(columns={'target_raw': 'target'})
print(f'清理后: {df.shape}')
print(f'target: mean={df["target"].mean():.6f}, std={df["target"].std():.6f}')

清理后: (15984, 40)
target: mean=0.001973, std=0.036916


In [4]:
# 采样加速（可选，去掉这行用全量15984）
# df = df.sample(n=3000, random_state=42)
print(f'训练集: {df.shape}')

训练集: (15984, 40)


In [5]:
# PyCaret setup
exp = setup(
    data=df,
    target='target',
    train_size=0.8,
    preprocess=True,  # 因子已预处理
    session_id=42,
    verbose=False,
    html=False,
)
print('setup 完成')

setup 完成


In [9]:
# compare_models: 一次性对比多个模型
# PyCaret 3.3.2 可用 ID: lr, lasso, ridge, en, lar, llar, omp, br, ard, par, ransac, tr,
#                        huber, kr, svm, knn, dt, rf, et, ada, gbr, mlp, lightgbm
best = compare_models(
    include=['ridge', 'lasso', 'en', 'huber', 'rf', 'et', 'gbr', 'lightgbm', 'ada'],
    sort='R2',
    n_select=5,  # 选前5个
    verbose=True,
)


Processing:   0%|          | 0/45 [00:00<?, ?it/s]
                                                           [A
Processing: 100%|██████████| 45/45 [15:44<00:00, 20.26s/it]
                                                           

                                    Model     MAE     MSE    RMSE      R2  \
et                  Extra Trees Regressor  0.0161  0.0006  0.0246  0.5582   
rf                Random Forest Regressor  0.0177  0.0007  0.0256  0.5223   
lightgbm  Light Gradient Boosting Machine  0.0204  0.0008  0.0286  0.4054   
gbr           Gradient Boosting Regressor  0.0235  0.0011  0.0325  0.2316   
ridge                    Ridge Regression  0.0265  0.0014  0.0369  0.0105   
lasso                    Lasso Regression  0.0266  0.0014  0.0371 -0.0014   
en                            Elastic Net  0.0266  0.0014  0.0371 -0.0014   
huber                     Huber Regressor  0.0266  0.0014  0.0372 -0.0036   
ada                    AdaBoost Regressor  0.0283  0.0014  0.0375 -0.0239   

           RMSLE    MAPE  TT (Sec)  
et        0.0199  1.2952     0.478  
rf        0.0216  1.3107     2.218  
lightgbm  0.0242  1.3976    88.888  
gbr       0.0287  1.2809     1.035  
ridge     0.0328  1.2055     0.143  
lasso  

In [7]:
# 查看评分表
results = exp.pull()
print(results.to_string())

                    Description             Value
0                    Session id                42
1                        Target            target
2                   Target type        Regression
3           Original data shape       (15984, 40)
4        Transformed data shape       (15984, 40)
5   Transformed train set shape       (12787, 40)
6    Transformed test set shape        (3197, 40)
7              Numeric features                39
8                    Preprocess              True
9               Imputation type            simple
10           Numeric imputation              mean
11       Categorical imputation              mode
12               Fold Generator             KFold
13                  Fold Number                10
14                     CPU Jobs                -1
15                      Use GPU             False
16               Log Experiment             False
17              Experiment Name  reg-default-name
18                          USI              7624


In [8]:
# 提取 top-5 模型类型
if isinstance(best, list):
    top5_ids = [type(m).__name__ for m in best]
    print(f'Top-5 模型: {top5_ids}')
else:
    print(f'Best 模型: {type(best).__name__}')

NameError: name 'best' is not defined

In [ ]:
# 特征重要性（如果是树模型）
feat_cols_target = [c for c in df.columns if c.startswith('f')]
for m in (best if isinstance(best, list) else [best]):
    name = type(m).__name__
    if hasattr(m, 'feature_importances_'):
        imp = pd.Series(m.feature_importances_, index=feat_cols_target).sort_values(ascending=False)
        print(f'\n{name} Top-10 特征:')
        print(imp.head(10).to_string())

In [ ]:
# 可视化: 模型对比
exp.plot_model(best[0] if isinstance(best, list) else best, plot='error')

In [ ]:
# 可视化: 特征重要性
exp.plot_model(best[0] if isinstance(best, list) else best, plot='feature')